## Player Overperformance by Position

### Project description

This project analyzes wether player overperformance can be predicted while taking positional context into account.

In football, player output should not be evaluated equally across all positions. Forwards are generally expected to score more than midfielders or defenders, so position can provide important context when analyzing efficiency and overperformance.

The goal of this project is to build a binary classification model that predicts wether a player overperformed relative to expected goals (xG), while incorporating position into the analysis.

### Objective

Develop a machine learning model to classify players into:

- **1 = overperformed**
- **2 = did not overperform**

### Main hypothesis

Including positional context may improve the interpretation of overperformance patterns and lead to and evaluation of player performance.


In [12]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [13]:
# load dataset
df = pd.read_csv('../data/players.csv')

df.head()

,Squad,Comp,Player,Nation,Pos,Age,Born,MP,Minutes_Played,Mn_per_MP,...,Launch_percent_Passes_GK,AvgLen_Passes_GK,Att_Goal_Kicks_GK,AvgLen_GoalKick_GK,CrossesFaced_GK,CrossedStopped_GK,CrossedStopped_Perc_GK,OPA_Sweeper_GK,OPA_per_90_Sweeper_GK,AvgDist_Sweeper_GK
0,Ajaccio,Ligue 1,Mickaël Alphonse,GLP,DF,33,1989,26,1361.0,52.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ajaccio,Ligue 1,Cédric Avinel,GLP,DF,35,1986,24,1794.0,75.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ajaccio,Ligue 1,Mickaël Barreto,FRA,MF,31,1991,20,1325.0,66.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Ajaccio,Ligue 1,Cyrille Bayala,BFA,MF,26,1996,33,2009.0,61.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ajaccio,Ligue 1,Youcef Belaïli,ALG,"MF,FW",30,1992,17,1150.0,68.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Initial data inspection

Before building the model, the dataset is inspected to understand its structure, data types, missing values, and duplicated records. This step helps identify what cleaning and preparation steps are required before creating the target variable and selecting features.

In [14]:
# initial data analysis
df.shape

(3317, 145)

In [15]:
# column names
df.columns.tolist()

['Squad',
 'Comp',
 'Player',
 'Nation',
 'Pos',
 'Age',
 'Born',
 'MP',
 'Minutes_Played',
 'Mn_per_MP',
 'Mins_Per_90',
 'Starts',
 'PPM_Team.Success',
 'onG_Team.Success',
 'onGA_Team.Success',
 'plus_per__minus__Team.Success',
 'Goals',
 'Assists',
 'GoalsAssistsCombined',
 'NonPKG',
 'PK',
 'PKatt',
 'CrdY',
 'CrdR',
 'xG',
 'xAG',
 'npxG+xAG',
 'PrgC',
 'PrgP',
 'Gls_Per90',
 'Ast_Per90',
 'G+A_Per90',
 'G_minus_PK_Per',
 'G+A_minus_PK_Per',
 'xG_Per',
 'xAG_Per',
 'xG+xAG_Per',
 'Shots',
 'Shots_on_Target',
 'SoT_percent',
 'G_per_Sh',
 'G_per_SoT',
 'Avg_Shot_Dist',
 'FK_Standard',
 'G_minus_xG_Expected',
 'np:G_minus_xG_Expected',
 'Passes_Completed',
 'Passes_Attempted',
 'Passes_Cmp_percent',
 'PrgDist_Total',
 'Passes_Cmp_Short',
 'Passes_Att_Short',
 'Passes_Cmp_percent_Short',
 'Passes_Cmp_Medium',
 'Passes_Att_Medium',
 'Passes_Cmp_percent_Medium',
 'Passes_Cmp_Long',
 'Passes_Att_Long',
 'Passes_Cmp_percent_Long',
 'xA_Expected',
 'A_minus_xAG_Expected',
 'Key_Passes',


In [16]:
# general info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3317 entries, 0 to 3316
Columns: 145 entries, Squad to AvgDist_Sweeper_GK
dtypes: float64(136), int64(4), object(5)
memory usage: 3.7+ MB


In [17]:
# missing values
df.isna().sum()

Squad                        0
Comp                         0
Player                       0
Nation                       4
Pos                          0
                          ... 
CrossedStopped_GK         3110
CrossedStopped_Perc_GK    3111
OPA_Sweeper_GK            3110
OPA_per_90_Sweeper_GK     3110
AvgDist_Sweeper_GK        3115
Length: 145, dtype: int64

In [18]:
# duplicated rows
df.duplicated().sum()

np.int64(0)

### Initial data inspection

The dataset was reviewed to understand its structure, dimensions, column names, data types, missing values, and duplicated records. This initial inspection helps identify the cleaning and preparation steps needed before defining the target variable and selecting model features.

In [19]:
# check for 0 in xG
(df['xG'] == 0).sum()

np.int64(595)

In [20]:
# remove players with xG=0
df = df[df['xG'] > 0].copy()

# check new shape
df.shape

(2123, 145)

### Handling zero xG values

Players with xG equal to zero were removed from the dataset. Since the target variable is based on the ratio between goals and expected goals, these cases would lead to undefined or misleading efficiency values.

Additionally, players with zero xG typically have very limited attacking involvement, making them less relevant for this type of analysis.

In [21]:
# create efficiency
df['efficiency'] = df['Goals'] / df['xG']

# preview
df[['Player', 'Pos', 'Goals', 'xG', 'efficiency']].head()

,Player,Pos,Goals,xG,efficiency
0,Mickaël Alphonse,DF,0.0,0.5,0.000000
1,Cédric Avinel,DF,1.0,1.0,1.000000
2,Mickaël Barreto,MF,1.0,0.9,1.111111
3,Cyrille Bayala,MF,1.0,2.0,0.500000
4,Youcef Belaïli,"MF,FW",6.0,6.2,0.967742


In [22]:
# keep only first position
df['Pos'] = df['Pos'].str.split(',').str[0]

# check result
df['Pos'].value_counts()

Pos
DF    806
MF    734
FW    578
GK      5
Name: count, dtype: int64

### Position cleaning

Some players were assigned multiple positions (e.g., "MF,FW"). To simplify the analysis and ensure consistency, only the primary position (the first listed) was retained.

This allows the model to learn clearer patterns associated with each role.